# Prepare Img2GPS3K bounding boxes with Grounding DINO

This notebook reads the Img2GPS3K image directory, its metadata JSON, and a line-based vocabulary file. It runs Grounding DINO over the complete dataset and writes a manifest consumed directly by `geoclip_group_random_search_colab.ipynb`.

Bounding boxes remain in original-image pixel coordinates. CLIP resize/center-crop mapping belongs to the downstream GeoCLIP notebook.

In [ ]:
%pip install -q "transformers==5.14.1"

## 1. Configuration

Use `grounding-dino-tiny` for a quick smoke test, then switch back to `grounding-dino-base` for the dataset run. Set `LIMIT` to a small integer for testing and `None` for all images.

In [ ]:
import hashlib
import json
import math
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from PIL import Image, ImageDraw
from tqdm.auto import tqdm

DATASET_DIR = Path('/content/dataset/img2gps3k')
IMAGES_DIR = DATASET_DIR / 'images'
METADATA_JSON = DATASET_DIR / 'im2gps3k_metadata.json'
VOCAB_TXT = Path('/content/geolocation_vocab.txt')
OUTPUT_DIR = DATASET_DIR / 'grounding_dino_output'

MODEL_ID = 'IDEA-Research/grounding-dino-base'
BOX_THRESHOLD = 0.4
TEXT_THRESHOLD = 0.3
LIMIT = None
QA_SAMPLES = 9
SEED = 42

CHECKPOINT_JSONL = OUTPUT_DIR / 'checkpoint.jsonl'
MANIFEST_JSON = OUTPUT_DIR / 'manifest.json'
SUMMARY_JSON = OUTPUT_DIR / 'run_summary.json'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
print('Device:', DEVICE)

## 2. Validate dataset and vocabulary

The metadata `image_path` field is intentionally ignored. Images are matched by metadata key to `images/<key>.jpg`, avoiding the stale `im2gps3k` path stored in the source metadata.

In [ ]:
def load_vocabulary(path):
    vocabulary = []
    seen = set()
    for raw_line in Path(path).read_text(encoding='utf-8').splitlines():
        phrase = raw_line.strip()
        if not phrase or phrase.startswith('#') or phrase in seen:
            continue
        seen.add(phrase)
        vocabulary.append(phrase)
    if not vocabulary:
        raise ValueError('Vocabulary must contain at least one non-comment line')
    return vocabulary


def load_dataset_index(metadata_path, images_dir):
    metadata = json.loads(Path(metadata_path).read_text(encoding='utf-8'))
    if not isinstance(metadata, dict) or not metadata:
        raise ValueError('Metadata must be a non-empty JSON object keyed by image id')
    image_paths = {path.stem: path for path in Path(images_dir).glob('*.jpg')}
    metadata_ids, image_ids = set(metadata), set(image_paths)
    missing = sorted(metadata_ids - image_ids)
    if missing:
        raise ValueError(f'{len(missing)} metadata images are missing; first IDs: {missing[:10]}')
    extra = sorted(image_ids - metadata_ids)
    if extra:
        print(f'Ignoring {len(extra)} images without metadata; first IDs: {extra[:10]}')

    records = []
    for image_id in sorted(metadata):
        coords = metadata[image_id].get('gt_coords')
        if not isinstance(coords, list) or len(coords) != 2:
            raise ValueError(f'{image_id}: gt_coords must be [lat, lon]')
        lat, lon = map(float, coords)
        if not (math.isfinite(lat) and -90 <= lat <= 90 and math.isfinite(lon) and -180 <= lon <= 180):
            raise ValueError(f'{image_id}: invalid ground-truth coordinates {coords}')
        records.append({'id': image_id, 'image_path': image_paths[image_id], 'lat': lat, 'lon': lon})
    return records


vocabulary = load_vocabulary(VOCAB_TXT)
dataset_records = load_dataset_index(METADATA_JSON, IMAGES_DIR)
if LIMIT is not None:
    dataset_records = dataset_records[:int(LIMIT)]
print(f'Images selected: {len(dataset_records):,}')
print(f'Vocabulary ({len(vocabulary)}):', vocabulary)

## 3. Load Grounding DINO

In [ ]:
from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor

processor = AutoProcessor.from_pretrained(MODEL_ID)
detector = AutoModelForZeroShotObjectDetection.from_pretrained(MODEL_ID).to(DEVICE).eval()
MODEL_COMMIT = getattr(detector.config, '_commit_hash', None)
print('Loaded:', MODEL_ID, MODEL_COMMIT or '')

## 4. Detection and resumable checkpoint

One JSON line is flushed after every image. Rerunning the cell skips completed `ok` and `no_boxes` records while retrying previous errors.

In [ ]:
def clamp_box(box, width, height):
    x1, y1, x2, y2 = map(float, box)
    x1, x2 = min(max(x1, 0.0), width), min(max(x2, 0.0), width)
    y1, y2 = min(max(y1, 0.0), height), min(max(y2, 0.0), height)
    if x2 <= x1 or y2 <= y1:
        return None
    return [round(x1, 4), round(y1, 4), round(x2, 4), round(y2, 4)]


@torch.inference_mode()
def detect_objects(image):
    text_labels = [vocabulary]
    inputs = processor(images=image, text=text_labels, return_tensors='pt').to(DEVICE)
    outputs = detector(**inputs)
    result = processor.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        threshold=BOX_THRESHOLD,
        text_threshold=TEXT_THRESHOLD,
        target_sizes=[image.size[::-1]],
    )[0]

    width, height = image.size
    detections = []
    labels = result.get('text_labels')
    if labels is None:
        labels = result.get('labels', [])
    for box, score, label in zip(result['boxes'], result['scores'], labels):
        box = clamp_box(box.detach().cpu().tolist(), width, height)
        if box is not None:
            detections.append({'label': str(label), 'score': float(score.detach().cpu()), 'box': box})
    return sorted(detections, key=lambda item: item['score'], reverse=True)


def read_checkpoint(path):
    latest = {}
    path = Path(path)
    if not path.exists():
        return latest
    for line_number, line in enumerate(path.read_text(encoding='utf-8').splitlines(), 1):
        if not line.strip():
            continue
        try:
            entry = json.loads(line)
            latest[entry['id']] = entry
        except (json.JSONDecodeError, KeyError) as error:
            raise ValueError(f'Invalid checkpoint line {line_number}: {error}') from error
    return latest


def append_checkpoint(path, entry):
    with Path(path).open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(entry, ensure_ascii=False) + '\n')
        handle.flush()


assert clamp_box([-1, -2, 20, 30], 10, 10) == [0.0, 0.0, 10.0, 10.0]
assert clamp_box([5, 5, 5, 9], 10, 10) is None

run_config = {
    'model_id': MODEL_ID,
    'model_commit': MODEL_COMMIT,
    'box_threshold': BOX_THRESHOLD,
    'text_threshold': TEXT_THRESHOLD,
    'vocabulary': vocabulary,
}
RUN_SIGNATURE = hashlib.sha256(json.dumps(run_config, sort_keys=True).encode()).hexdigest()[:16]
checkpoint = read_checkpoint(CHECKPOINT_JSONL)
checkpoint_signatures = {entry.get('run_signature') for entry in checkpoint.values()}
if checkpoint_signatures and checkpoint_signatures != {RUN_SIGNATURE}:
    raise ValueError(f'Checkpoint belongs to a different configuration: {checkpoint_signatures}. Use a new OUTPUT_DIR.')
completed_ids = {image_id for image_id, entry in checkpoint.items() if entry.get('status') in {'ok', 'no_boxes'}}
pending = [record for record in dataset_records if record['id'] not in completed_ids]
print(f'Resuming with {len(completed_ids):,} complete and {len(pending):,} pending')

for record in tqdm(pending, desc='Grounding DINO'):
    try:
        with Image.open(record['image_path']) as source_image:
            image = source_image.convert('RGB')
        width, height = image.size
        detections = detect_objects(image)
        relative_image = Path(os.path.relpath(record['image_path'], OUTPUT_DIR)).as_posix()
        status = 'ok' if detections else 'no_boxes'
        entry = {
            'id': record['id'],
            'image': relative_image,
            'boxes': [item['box'] for item in detections],
            'ground_truth': {'lat': record['lat'], 'lon': record['lon']},
            'detections': detections,
            'detection_status': status,
            'image_size': {'width': width, 'height': height},
            'status': status,
            'run_signature': RUN_SIGNATURE,
        }
    except KeyboardInterrupt:
        raise
    except Exception as error:
        entry = {'id': record['id'], 'status': 'error', 'error': f'{type(error).__name__}: {error}', 'run_signature': RUN_SIGNATURE}
    append_checkpoint(CHECKPOINT_JSONL, entry)

print('Checkpoint complete for this run')

## 5. Build the GeoCLIP manifest and summary

Records with no detections remain in the manifest with `boxes: []`. The downstream notebook reports and skips records that produce an empty CLIP patch mask. Error records are excluded until a later resume succeeds.

In [ ]:
checkpoint = read_checkpoint(CHECKPOINT_JSONL)
selected_ids = [record['id'] for record in dataset_records]
completed = [checkpoint[image_id] for image_id in selected_ids if checkpoint.get(image_id, {}).get('status') in {'ok', 'no_boxes'}]
errors = [checkpoint[image_id] for image_id in selected_ids if checkpoint.get(image_id, {}).get('status') == 'error']

manifest = []
for entry in completed:
    manifest.append({key: value for key, value in entry.items() if key not in {'status', 'error', 'run_signature'}})
MANIFEST_JSON.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

summary = {
    'run_signature': RUN_SIGNATURE,
    'model_id': MODEL_ID,
    'model_commit': MODEL_COMMIT,
    'box_threshold': BOX_THRESHOLD,
    'text_threshold': TEXT_THRESHOLD,
    'vocabulary': vocabulary,
    'requested_images': len(dataset_records),
    'manifest_records': len(manifest),
    'with_boxes': sum(bool(entry['boxes']) for entry in manifest),
    'without_boxes': sum(not entry['boxes'] for entry in manifest),
    'errors': len(errors),
    'total_boxes': sum(len(entry['boxes']) for entry in manifest),
    'checkpoint': str(CHECKPOINT_JSONL),
    'manifest': str(MANIFEST_JSON),
}
SUMMARY_JSON.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
display(summary)
if errors:
    print('Errors will be retried next run:', errors[:5])

## 6. Visual quality check

In [ ]:
records_with_boxes = [entry for entry in manifest if entry['boxes']]
sample_count = min(QA_SAMPLES, len(records_with_boxes))
if sample_count:
    samples = random.Random(SEED).sample(records_with_boxes, sample_count)
    columns = 3
    rows = math.ceil(sample_count / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(16, 5 * rows))
    axes = list(getattr(axes, 'flat', [axes]))
    for axis, entry in zip(axes, samples):
        image_path = (OUTPUT_DIR / entry['image']).resolve()
        with Image.open(image_path) as source_image:
            image = source_image.convert('RGB')
        draw = ImageDraw.Draw(image)
        for detection in entry['detections']:
            draw.rectangle(detection['box'], outline='red', width=4)
            draw.text((detection['box'][0], detection['box'][1]), f"{detection['label']} {detection['score']:.2f}", fill='red')
        axis.imshow(image)
        axis.set_title(entry['id'])
        axis.axis('off')
    for axis in axes[sample_count:]:
        axis.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No detected boxes available for visual QA')